In [1]:
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import colors
from scipy.interpolate import griddata, RegularGridInterpolator
import pygeode as pyg
import os, glob
from jfunctions import *
from mpas_funcs import *

%matplotlib widget

save = False
#save_dir='/home/cwp29/tmp/mpas/T_nudge/'

In [2]:
spinup_data_dir = '/data/phh1-hosted/cwp29/mpas-a/nudge_test/'
data_dir = '/data/phh1-hosted/cwp29/mpas-a/T_nudge_test/'
grid_path = data_dir + 'x1.40962.grid.nc'

config = get_config(data_dir)
nudge_u = config['config_nudge_u']
nudge_T = config['config_nudge_T']

dsd = pyg.open(data_dir+'latlon_diag.nc')
dsh = pyg.open(data_dir+'latlon_hist.nc')

pre_dsd = pyg.open(spinup_data_dir+'latlon_diag.nc')
pre_dsh = pyg.open(spinup_data_dir+'latlon_hist.nc')

print(dsd)
print(dsh)

<Dataset>:
Vars:
  t_isobaric           (Time,nIsoLevelsT,latitude,longitude)  (3,22,180,360)
  z_isobaric           (Time,nIsoLevelsZ,latitude,longitude)  (3,22,180,360)
  omg_isobaric         (Time,nIsoLevelsZ,latitude,longitude)  (3,22,180,360)
  rh_isobaric          (Time,nIsoLevelsT,latitude,longitude)  (3,22,180,360)
  qv_isobaric          (Time,nIsoLevelsT,latitude,longitude)  (3,22,180,360)
  uzonal_isobaric      (Time,nIsoLevelsU,latitude,longitude)  (3,22,180,360)
  umeridional_isobaric (Time,nIsoLevelsU,latitude,longitude)  (3,22,180,360)
  olrtoa               (Time,latitude,longitude)              (3,180,360)
  i_rainc              (Time,latitude,longitude)              (3,180,360)
  i_rainnc             (Time,latitude,longitude)              (3,180,360)
  rainc                (Time,latitude,longitude)              (3,180,360)
  rainnc               (Time,latitude,longitude)              (3,180,360)
  precipw              (Time,latitude,longitude)              (3,180,360)


In [3]:
lats = dsd.latitude[:]
lons = dsd.longitude[:]

zs = 0.5*(dsh.zgrid[1:, 0, 0] + dsh.zgrid[:-1, 0, 0])
zs_hres = np.linspace(0, np.max(zs), 256)

ps = dsh.pressure[0, :, 0, 0]

In [ ]:
# Time and zonal mean zonal velocities

Tbar = dsd.t_isobaric.mean('longitude', 'Time')[:]
ubar_iso = dsd.uzonal_isobaric.mean('longitude', 'Time')[:]
ubar = dsd.uReconstructZonal.mean('longitude', 'Time')[:]
thetabar = dsh.theta.mean('longitude', 'Time')[:]

nudge_ubar = dsd.nudge_uRZonal.mean('longitude', 'Time')[:]
tend_ubar = dsd.tend_uRZonal.mean('longitude', 'Time')[:]

nudge_thetabar = dsd.theta_nudge.mean('longitude', 'Time')[:]
tend_thetabar = dsd.nudge_tend_theta.mean('longitude', 'Time')[:]

presbar = dsh.pressure.mean('longitude', 'Time')[:]

ndays = 7.
dt = 6.
idx = int(ndays * 24./dt)
pre_ubar = pre_dsh.uReconstructZonal(i_Time=(-1-idx, -1)).mean('longitude', 'Time')[:]
pre_thetabar = pre_dsh.theta(i_Time=(-1-idx, -1)).mean('longitude', 'Time')[:]
pre_presbar = pre_dsh.pressure(i_Time=(-1-idx, -1)).mean('longitude', 'Time')[:]

In [ ]:
# Convert potential temperature to temperature
# theta = T * (p0/p)^(R/c_p)
# T = theta * (p/p0)^(R/c_p)
R_cp = 0.286
p0 = 1000e2 # hPa

if nudge_T:
    #temp = dsh.theta * (dsh.pressure / p0)**R_cp
    Tbar = thetabar * (presbar / p0)**R_cp
    
    nudge_Tbar = nudge_thetabar * (presbar / p0)**R_cp
    tend_Tbar = tend_thetabar * (presbar / p0)**R_cp

    pre_Tbar = pre_thetabar * (pre_presbar / p0)**R_cp

In [ ]:
# Nudging kernels
zntop = 18000
znbtm = 16000
znrlx = 1500
L_lat = 15

k_vert = lambda z: np.exp(-np.power((z - 17000)/znrlx, 2))

k_horiz = lambda l: np.exp(-np.power(l/L_lat, 2))

U = 10.
T = 200.

if nudge_u:
    f_nudge = lambda l, z: U * k_vert(z) * k_horiz(l)
elif nudge_T:
    f_nudge = lambda l, z: T * k_vert(z) * k_horiz(l)

In [ ]:
# Utility functions for converting between altitude and pressure

def alt_to_pres(z, standard=False):
    if standard:
        p_0 = 1.01e5
        H = 8.5e3
    
        return p_0 * np.exp(-z/H)
    else:
        return np.interp(z, zs, ps)
        
def pres_to_alt(p):
    return np.interp(p, ps, zs)

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(8, 7), constrained_layout=True)

pres = np.array([100, 200, 300, 500, 700, 1000, 2000, 3000, 5000, 7000, 10000, 15000, 20000, 25000, 30000, 40000, 50000, 60000, 70000, 85000, 92500, 100000], dtype=np.float64)
pres /= 100

"""
# T shaded with u contours
T_im = ax[0, 0].contourf(lats, pres, Tbar, levels=np.arange(190, 305, 5), extend='both', cmap='coolwarm')
u_im_neg = ax[0, 0].contour(lats, pres, ubar_iso, levels=np.arange(-20, 0, 5), colors='k', linestyles=':', linewidths=0.5)
u_im_zero = ax[0, 0].contour(lats, pres, ubar_iso, levels=[0], colors='k', linewidths=1)
u_im_pos = ax[0, 0].contour(lats, pres, ubar_iso, levels=np.arange(5, 55, 5), colors='k', linewidths=0.5)
ax[0, 0].clabel(u_im_pos, u_im_pos.levels, fontsize=6)
ax[0, 0].clabel(u_im_neg, u_im_neg.levels, fontsize=6)
fig.colorbar(T_im, ax=ax[0, 0], location='bottom', label="temperature (K)")
"""

nudge_target = U*np.outer(k_vert(zs), k_horiz(lats))
ulevs = np.linspace(-50, 50, 21)

# control (end of spin up)
preU_im = ax[0, 0].contourf(lats, zs, pre_ubar, levels=ulevs, cmap='coolwarm',extend='both')
ax[0, 0].set_title("Spin-up")
#fig.colorbar(preU_im, ax=ax[0, 0], label="Control u", location="bottom")

# u shaded
print(ubar.max(), ubar.min())
U_im = ax[0, 1].contourf(lats, zs, ubar, levels=ulevs, cmap='coolwarm', extend='both')
ax[0, 1].set_title("Nudged")
fig.colorbar(U_im, ax=ax[0, 1], label="Zonal velocity (m/s)", location="right")

# nudging tendency shaded
tend_im = ax[1, 0].contourf(lats, zs, tend_ubar, levels=16, cmap='bwr', norm=colors.CenteredNorm(), extend='both')
fig.colorbar(tend_im, ax=ax[1, 0], label="Nudging tendency", location="bottom")

# nudging kernel timescale
print(nudge_ubar.max(), nudge_ubar.min())
Unudge_im = ax[1, 1].contourf(lats, zs, (10./nudge_ubar)/(86400), cmap='hot', norm=colors.LogNorm(), levels=np.power(10, np.linspace(-2, 2, 9)), extend='both')
fig.colorbar(Unudge_im, ax=ax[1, 1], label="Effective nudging timescale (days)", location="bottom")

# kernels
vscale = 10
hscale = 2e3
ax[1, 1].plot(lats, 0.5*(zntop+znbtm) + hscale*k_horiz(lats), color='b', label="horizontal kernel")
ax[1, 1].plot(vscale*k_vert(zs), zs, color='r', label="vertical kernel")
ax[1, 1].legend()

for a in ax.ravel()[:-1]:
    nudge_im_neg = a.contour(lats, zs, nudge_target, levels=np.arange(-11, 1, 2)[:-1], colors='r', linestyles=':', linewidths=0.5)
    #a.contour(lats, zs, nudge_target, levels=[0], colors='r', linewidths=1)
    nudge_im_pos = a.contour(lats, zs, nudge_target, levels=np.arange(0, 10, 2)[1:], colors='r', linewidths=0.5)

    a.clabel(nudge_im_pos, nudge_im_pos.levels, fontsize=6)
    a.clabel(nudge_im_neg, nudge_im_neg.levels, fontsize=6)

for a in ax.ravel():
    a.set_xlim(-90, 90)
    a.set_xlabel("Latitude (degrees)")

ax[0, 0].set_ylabel("Height (m)")

if save: save_fig(fig, save_dir+'u_nudge.png', dpi=300)

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(8, 7), constrained_layout=True)

pres = np.array([100, 200, 300, 500, 700, 1000, 2000, 3000, 5000, 7000, 10000, 15000, 20000, 25000, 30000, 40000, 50000, 60000, 70000, 85000, 92500, 100000], dtype=np.float64)
pres /= 100

"""
# T shaded with u contours
T_im = ax[0, 0].contourf(lats, pres, Tbar, levels=np.arange(190, 305, 5), extend='both', cmap='coolwarm')
u_im_neg = ax[0, 0].contour(lats, pres, ubar_iso, levels=np.arange(-20, 0, 5), colors='k', linestyles=':', linewidths=0.5)
u_im_zero = ax[0, 0].contour(lats, pres, ubar_iso, levels=[0], colors='k', linewidths=1)
u_im_pos = ax[0, 0].contour(lats, pres, ubar_iso, levels=np.arange(5, 55, 5), colors='k', linewidths=0.5)
ax[0, 0].clabel(u_im_pos, u_im_pos.levels, fontsize=6)
ax[0, 0].clabel(u_im_neg, u_im_neg.levels, fontsize=6)
fig.colorbar(T_im, ax=ax[0, 0], location='bottom', label="temperature (K)")
"""

nudge_target = T*np.outer(k_vert(zs), k_horiz(lats))
Tlevs = np.arange(190, 305, 5)

# control (end of spin up)
preT_im = ax[0, 0].contourf(lats, zs, pre_Tbar, levels=Tlevs, cmap='coolwarm',extend='both')
ax[0, 0].set_title("Spin-up")
#fig.colorbar(preT_im, ax=ax[0, 0], label="Control u", location="bottom")

# u shaded
print(ubar.max(), ubar.min())
T_im = ax[0, 1].contourf(lats, zs, Tbar, levels=Tlevs, cmap='coolwarm', extend='both')
ax[0, 1].set_title("Nudged")
fig.colorbar(T_im, ax=ax[0, 1], label="Temperature (K)", location="right")

# nudging tendency shaded
tend_im = ax[1, 0].contourf(lats, zs, tend_Tbar, levels=16, cmap='bwr', norm=colors.CenteredNorm())
fig.colorbar(tend_im, ax=ax[1, 0], label="Nudging tendency", location="bottom")

# nudging kernel timescale
print(nudge_Tbar.max(), nudge_Tbar.min())

Tnudge_im = ax[1, 1].contourf(lats, zs, (10./nudge_Tbar)/(86400), cmap='hot', norm=colors.LogNorm(), levels=np.power(10, np.linspace(-2, 2, 9)), extend='both')
fig.colorbar(Tnudge_im, ax=ax[1, 1], label="Effective nudging timescale (days)", location="bottom")

# kernels
vscale = 10
hscale = 2e3
ax[1, 1].plot(lats, 0.5*(zntop+znbtm) + hscale*k_horiz(lats), color='b', label="horizontal kernel")
ax[1, 1].plot(vscale*k_vert(zs), zs, color='r', label="vertical kernel")
ax[1, 1].legend()

for a in ax.ravel()[:-1]:
    nudge_im_pos = a.contour(lats, zs, nudge_target, levels=np.arange(0, 300, 50)[1:], colors='r', linewidths=0.5)

    a.clabel(nudge_im_pos, nudge_im_pos.levels, fontsize=6)
    a.clabel(nudge_im_neg, nudge_im_neg.levels, fontsize=6)

for a in ax.ravel():
    a.set_xlim(-90, 90)
    a.set_xlabel("Latitude (degrees)")

ax[0, 0].set_ylabel("Height (m)")

if save: save_fig(fig, save_dir+'u_nudge.png', dpi=300)